# Graphs


## Reading material

- [Chapter 5: Basic Graph Algorithms](https://jeffe.cs.illinois.edu/teaching/algorithms/book/05-graphs.pdf) —
  the vocabulary we'll build everything else on: what a graph actually is
  (vertices, edges, directed vs. undirected), how to represent one in code
  (adjacency lists vs. adjacency matrices, and the space/time tradeoffs
  between them), and the basic structural ideas — paths, cycles,
  connectivity, and trees — that the rest of the unit assumes you're
  comfortable with.
- [Chapter 7: Minimum Spanning Trees](https://jeffe.cs.illinois.edu/teaching/algorithms/book/07-mst.pdf) —
  given a connected, weighted graph, how do we find the cheapest set of
  edges that keeps every vertex connected? The chapter develops the
  cut property and cycle property that justify _why_ a greedy choice of
  edges works at all, then uses them to derive Boruvka's, Prim's, and
  Kruskal's algorithms.

Skim first, then read closely. The notation will feel unfamiliar at first —
that's normal for a new topic, not a sign you're behind. As with the
dynamic programming reading, the move is: read a definition or claim,
close the PDF, and try to restate it (or re-derive it) yourself before
looking again.

## The basics

A graph $G$ is a pair of two sets. A set of vertices, $V$ and a set of edges $E$ between some (or all) of its vertices. We write

$$
\begin{align}
G & = (V, E) \\
V & = \{ 1,2,\ldots, n\} \\
E &  = \{(i,j)\, |\, i,j \in V;\, i\neq j \}
\end{align}
$$

We operationalize this notation as an $n\times n$ array $g$:

$$
\begin{align}
g_{ij} =  \begin{cases} 0,\ \text{if } (i,j)\,\not\in E \\ 1,\ \text{if } (i,j)\,\in E\end{cases}
\end{align}
$$

As we'll see, 0 and 1 are not the only values used to denote the absence or presence of an edge. For now, they are sufficient.

### Two representations, one choice

A graph can be represented as an **adjacency list** (each vertex keeps a
list of its neighbors) or an **adjacency matrix** (the $n\times n$ array
$g$ above). An adjacency list looks cheaper at first glance -- a sparse
graph has far fewer edges than $n^2$ -- but each neighbor in that list has
to be an object with its own payload, pointers, and methods, and an
object costs far more memory than a single integer. The matrix trades a
little extra space (one entry for every *possible* edge, not just the
ones that exist) for representing each edge as a plain number. We will
use the adjacency matrix throughout this course.

### What a row or column tells you

For an undirected graph, $g$ is symmetric ($g_{ij} = g_{ji}$), so row $i$
and column $i$ carry the same information about vertex $i$'s
neighborhood:

- The number of nonzero entries in row/column $i$ is the **degree** of
  vertex $i$. (In a *directed* graph, the nonzero entries in row $i$ give
  the **out-degree** and those in column $i$ give the **in-degree** --
  and the two can differ.)
- A row that is all zeros means that vertex has no edges at all: it is
  disconnected from the rest of the graph.
- The maximum possible number of edges in a graph with $n$ vertices is
  $\binom{n}{2} = \frac{n(n-1)}{2}$, i.e., $O(n^2)$, reached only when
  every vertex is adjacent to every other vertex. Most graphs worth
  studying (road networks, the web, social graphs) are far **sparser**
  than that -- which is exactly why a running time reported as
  $O(|V|+|E|)$ can look deceptively fast: in the worst case, $|E|$ is
  still $O(|V|^2)$.


## Building graphs

The helper below builds a random undirected graph as an adjacency
matrix, so we have something to experiment with before moving to the
fixed example worked through in class.


In [14]:
from __future__ import annotations
import random


def create_random_undirected_graph(
    n: int, prob: float, no_edge: int, max_edge_wait: int
) -> list[list[int]]:
    """
    Create a random undirected graph represented as an adjacency matrix.

    Parameters
    ----------
    n : int
        The number of vertices in the graph.
    prob : float
        The probability of an edge existing between any two vertices.
    no_edge : int
        The value to represent no edge between two vertices in the
        adjacency matrix. This value will be used to indicate that
        there is no edge between two vertices.
    max_edge_wait : int
        The maximum value to represent an edge between two vertices
        in the adjacency matrix. This value will be used to indicate that
        there is an edge between two vertices. If max_edge_wait is 1,
        then the graph will be unweighted. If max_edge_wait is
        greater than 1, then the graph will be weighted with random
        weights between 1 and max_edge_wait.

    Returns
    -------
    list[list[int]]
        The adjacency matrix representing the random undirected graph.
    """
    # Initialize the adjacency matrix with no_edge values
    g = [[no_edge for _ in range(n)] for _ in range(n)]
    # Consider every vertex i
    for i in range(n):
        # We know that (i,i) is always no_edge. Also, because the graph is
        # undirected, we only need to consider j > i.
        for j in range(i + 1, n):
            # Flip a biased coin for the edge (i, j)
            if random.random() < prob:
                # Unweighted graphs (max_edge_wait == 1) just use that
                # fixed value; weighted graphs draw a random weight in
                # [1, max_edge_wait]
                this_edge = (
                    max_edge_wait
                    if max_edge_wait == 1
                    else random.randint(1, max_edge_wait)
                )
                g[i][j] = this_edge
                # In an undirected graph, the edge (i,j) is the same as (j,i)
                g[j][i] = this_edge
    # Done!
    return g

In [15]:
# 5 vertices, 50% edge probability, 0 marks "no edge", weights in 1-10
create_random_undirected_graph(5, 0.5, 0, 10)

[[0, 8, 4, 5, 8],
 [8, 0, 1, 6, 0],
 [4, 1, 0, 2, 2],
 [5, 6, 2, 0, 6],
 [8, 0, 2, 6, 0]]

## The graph from class

The adjacency matrix below is the six-vertex graph worked through in
lecture. It has three components: $\{0,1,2\}$, $\{3,4\}$, and
$\{5\}$ -- vertex 5 has no edges at all. A graph does not have to be
fully connected, and most realistic ones (road networks included) are
not.


In [16]:
# Adjacency matrix for the class example graph (6 vertices, 3 components)
g = [
    [0, 1, 1, 0, 0, 0], # neighbors of vertex 0
    [1, 0, 0, 0, 0, 0], # neighbors of vertex 1
    [1, 0, 0, 0, 0, 0], # neighbors of vertex 2
    [0, 0, 0, 0, 1, 0], # neighbors of vertex 3
    [0, 0, 0, 1, 0, 0], # neighbors of vertex 4
    [0, 0, 0, 0, 0, 0], # neighbors of vertex 5
]

## Reachability: who can vertex $u$ get to?

Given a starting vertex, we want every vertex reachable from it --
directly, or through a chain of edges. The strategy from lecture:

1. Keep a worklist (`gonext`) of vertices still to explore, seeded with
   the starting vertex.
2. Keep a `visited` list of vertices already explored, so we never
   revisit a vertex (and never loop forever).
3. Pop a vertex $u$ off the worklist, mark it visited, and scan row $u$
   of the adjacency matrix: every nonzero entry at column $v$ is a
   neighbor of $u$ that still needs exploring, so add it to the
   worklist.
4. Repeat until the worklist is empty; return `visited`.

Because `gonext.pop()` removes from the *end* of the list, neighbors are
explored last-in-first-out (depth-first) rather than breadth-first --
but for the question "can I reach vertex $v$ at all," the order doesn't
matter, only the final `visited` set does.


In [17]:
def reached(starting_vertex: int, g: list[list[int]]) -> list[int]:
    visited: list[int] = []  # vertices already explored
    gonext: list[int] = [starting_vertex]  # worklist, seeded with the start
    while len(gonext) > 0:
        u = gonext.pop()  # LIFO: explore the most recently found vertex next
        if u not in visited:
            visited.append(u)
            # row u of the adjacency matrix lists u's neighbors
            for v in range(len(g)):
                if g[u][v] != 0 and v not in visited:
                    gonext.append(v)  # found a new vertex to explore
    return visited


## Trying it out

Vertex 5 has no edges in the class example (row 5 is all zeros), so it
should form its own component: `reached` starting from vertex 5 should
return just `[5]`.


In [18]:
# Expect [5]: vertex 5 is isolated in the class example graph
reached(5, g)

[5]

## Next: counting components

Running `reached` from every vertex not yet marked as visited, and
counting how many times we have to start over, gives the number of
connected components in a graph -- exactly the question lecture posed
for the six-vertex example above (three components: $\{0,1,2\}$,
$\{3,4\}$, and $\{5\}$). We'll build that on top of `reached` next.


In [19]:
def count_components(g: list[list[int]]) -> int:
    """
    Count the number of connected components in an undirected graph.

    Parameters
    ----------
    g : list[list[int]]
        The adjacency matrix representing the undirected graph.

    Returns
    -------
    int
        The number of connected components in the graph.
    """

    components: int = 0
    marked: list[int] = []  # vertices already swept into some component
    # Try every vertex as a potential "new" starting point: a graph
    # doesn't have to be fully connected (vertex 5 in the class example
    # can't reach anything), so no single reached() call is guaranteed
    # to touch every vertex -- we have to give each one a chance to start
    # a component of its own.
    for starting_vertex in range(len(g)):
        if starting_vertex not in marked:
            # Not marked yet means nobody's reached() call has swept this
            # vertex in before, so we've found a vertex in a component we
            # haven't counted -- that's one more component.
            components += 1
            # Sweep in everything reachable from here -- the whole
            # component -- so none of its vertices re-trigger the count
            # above when the loop reaches them later.
            marked.extend(reached(starting_vertex, g))
    return components


In [20]:
# Walking S = 0..5 against the class example graph:
#   S=0: not marked -> component 1; reached(0, g) marks {0, 1, 2}
#   S=1: already marked (from S=0) -> skip
#   S=2: already marked (from S=0) -> skip
#   S=3: not marked -> component 2; reached(3, g) marks {3, 4}
#   S=4: already marked (from S=3) -> skip
#   S=5: not marked -> component 3; reached(5, g) marks {5}
print(count_components(g))  # Expect 3: {0,1,2}, {3,4}, {5}


3
